# C6-pytorch — Practice p13 — Solution


The NumPy and torch branches apply the same two affine maps and the
same inclusive threshold between them.  Explicit float64 conversion at
the library boundary makes the arithmetic directly comparable.


In [ ]:
import numpy as np
import torch
import torch.nn as nn

torch.set_default_dtype(torch.float64)

def affine_layer(x, W, b):
    return x @ W.T + b

def step_activation(z):
    return (z >= 0).astype(float)

class DenseLayer(nn.Module):
    """Session 2's pinned dense layer: weight (out, in), bias (out,)."""

    def __init__(self, weight, bias):
        super().__init__()
        self.weight = nn.Parameter(torch.as_tensor(weight), requires_grad=False)
        self.bias = nn.Parameter(torch.as_tensor(bias), requires_grad=False)

    def forward(self, x):
        return x @ self.weight.T + self.bias

class ThresholdGate(nn.Module):
    """Session 2's gate: 1 where x >= 0, else 0; owns no parameters."""

    def forward(self, x):
        return (x >= 0).to(x.dtype)


W1 = np.array([[1.0, 0.0], [0.0, 1.0], [1.0, 1.0]])
b1 = np.array([0.0, 0.0, -1.0])
W2 = np.array([[1.5, -1.0, 2.0]])
b2 = np.array([0.25])
X_np = np.array([[0.0, 0.0], [1.0, 0.0], [0.0, 1.0],
                 [2.0, 2.0], [-1.0, 3.0]])

H_np = step_activation(affine_layer(X_np, W1, b1))
out_np = affine_layer(H_np, W2, b2)

class TinyNet(nn.Module):
    def __init__(self, weight1, bias1, weight2, bias2):
        super().__init__()
        self.hidden = DenseLayer(weight1, bias1)
        self.gate = ThresholdGate()
        self.readout = DenseLayer(weight2, bias2)

    def forward(self, x):
        return self.readout(self.gate(self.hidden(x)))


net = TinyNet(torch.as_tensor(W1), torch.as_tensor(b1),
              torch.as_tensor(W2), torch.as_tensor(b2))
X_t = torch.from_numpy(X_np).to(torch.float64)
out_t = net(X_t)

out_t_np64 = out_t.numpy().astype(np.float64)
out_np64 = out_np.astype(np.float64)
gap = float(np.abs(out_t_np64 - out_np64).max())
param_names = sorted(name for name, _ in net.named_parameters())
n_tensors = len(param_names)

H_np, out_np, out_t, gap, param_names


Assigning each `DenseLayer` to an attribute of `TinyNet` registers it
as a submodule, so its `weight` and `bias` appear under dotted names.
The gate owns no `nn.Parameter`, so it contributes no parameter name.


### Answer check


In [ ]:
expected = np.array([[0.75], [2.75], [2.75], [2.75], [1.25]], dtype=np.float64)
assert np.array_equal(out_np64, expected)
assert np.array_equal(out_t_np64, expected)
assert gap < 1e-12  # cross-library float64 agreement; exact 0.0 is BLAS-sensitive
assert param_names == ["hidden.bias", "hidden.weight", "readout.bias", "readout.weight"]
assert n_tensors == 4
